# ⚡ EV Quantum Forge · Fusion IV: Neural Views (S6E9)

| Version | What it added | Blend OOF | Public LB |
|---|---|---:|---:|
| Fusion v1 | five non-blend recipes, one 5-fold partition | 0.946245 | 0.94643 |
| Fusion II | partition bagging (4 fold seeds) | 0.946398 | 0.94645 |
| Fusion III | RealMLP (PyTorch) next to the trees | 0.946423 | 0.94647 |
| **Fusion IV** | **three new neural views** | *this run* | |

Fusion III showed that the network is the part that still moves the blend. Its OOF Spearman with the
trees was 0.996, while the trees correlate 0.997 to 0.999 with each other. Fusion IV therefore spends
its whole GPU budget on **new neural components**. The eight bagged components that Fusion II and
Fusion III already trained are loaded from those notebooks' outputs; both are this author's public
notebooks.

| New component | Idea |
|---|---|
| **N2** | RealMLP (yekenot port), 4 epochs instead of 2. The best epoch was still the last one. |
| **N3** | wider RealMLP: 3 × 512 units, dropout 0.1, 3 epochs |
| **M** | RealMLP fed chinzorigtganbat's nested-TE + income-neighbourhood features: the same network, a completely different input view |

**Rules unchanged.** Everything is trained from the competition data and the public original dataset,
either here or in the two attached notebooks. No other author's predictions are read. Every new
component uses the same four fold partitions (seeds 42 / 45 / 48 / 51) as the loaded ones, so all
OOF vectors stay comparable. The blend ships only if Caruana hill-climbing on logits beats the best
single component on at least 4 of 5 held-out folds.

## 1 · Setup

In [ ]:
from __future__ import annotations

import gc
import glob
import json
import os
import subprocess
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.ndimage import convolve1d
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

T0 = time.time()
SMOKE = os.environ.get("FUSION_SMOKE", "0") == "1"      # tiny local run: code-path check only
TIME_BUDGET_S = float(os.environ.get("FUSION_BUDGET_H", "10.5")) * 3600
BLEND_RESERVE_S = 1200                                   # kept free for blending and export
NFOLD = 5
PARTITION_SEEDS = tuple(int(v) for v in os.environ.get("FUSION_PARTITIONS", "42,45,48,51").split(","))
THREADS = os.cpu_count() or 4
TARGET, ID = "Will_Buy_EV", "id"
CATS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level"]
NUMS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
OUT = Path("fusion_outputs")
OUT.mkdir(exist_ok=True)


def elapsed() -> str:
    return f"{(time.time() - T0) / 60:6.1f} min"


def remaining() -> float:
    return TIME_BUDGET_S - (time.time() - T0)


def find(name: str) -> str | None:
    for root in (os.environ.get("FUSION_DATA", ""), "/kaggle/input", "data", "../data"):
        if root and Path(root).exists():
            hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
            if hits:
                return hits[0]
    return None


def gpu_usable() -> bool:
    if os.environ.get("FUSION_FORCE_CPU") == "1":
        return False
    try:
        subprocess.check_output("nvidia-smi", stderr=subprocess.DEVNULL, shell=True)
        probe = np.random.rand(600, 4)
        xgb.XGBClassifier(n_estimators=5, max_depth=3, tree_method="hist", device="cuda",
                          verbosity=0).fit(probe, (probe[:, 0] > 0.5).astype(int))
        return True
    except Exception as exc:  # noqa: BLE001 - any failure means CPU fallback
        print(f"GPU not usable for XGBoost ({type(exc).__name__}); XGBoost runs on CPU")
        return False


GPU = gpu_usable()
XGB_DEVICE = "cuda" if GPU else "cpu"

# Round caps. Smoke mode only checks the code paths.
R = dict(xgb=200, lgb=200, es=20, cat=40, fixed_scale=0.03) if SMOKE else \
    dict(xgb=100_000, lgb=20_000, es=500, cat=None, fixed_scale=1.0)

## 2 · Data and the fold partitions

In [ ]:
train_raw = pd.read_csv(find("train.csv"))
test_raw = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))
orig_path = find("EV_Adoption_and_Range_Anxiety_Dataset.csv")
orig_raw = pd.read_csv(orig_path) if orig_path else None

if SMOKE:
    train_raw = train_raw.sample(40_000, random_state=0).reset_index(drop=True)
    test_raw = test_raw.head(15_000).reset_index(drop=True)
    sample = sample[sample[ID].isin(test_raw[ID])].reset_index(drop=True)
    if orig_raw is None:  # local stand-in so every code path runs; never used on Kaggle
        orig_raw = pd.read_csv(find("train.csv"), nrows=8_000).rename(columns={ID: "Buyer_ID"})
        print("SMOKE: original dataset missing, using a train slice as a stand-in")

y = (train_raw[TARGET] == "Yes").astype(np.int8).to_numpy()
test_ids = test_raw[ID].to_numpy()
if SMOKE:
    PARTITION_SEEDS = PARTITION_SEEDS[:2]
FOLDS_BY_SEED = {seed: list(StratifiedKFold(NFOLD, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))
                 for seed in PARTITION_SEEDS}
GATE_FOLDS = FOLDS_BY_SEED[PARTITION_SEEDS[0]]      # partition 1 folds drive the nested blend gate
fold_id = np.empty(len(y), dtype=np.int8)
for k, (_, va) in enumerate(GATE_FOLDS):
    fold_id[va] = k

print(f"train {train_raw.shape} | test {test_raw.shape} | original "
      f"{None if orig_raw is None else orig_raw.shape} | positives {y.mean():.4%}")
print(f"GPU for XGBoost: {GPU} | threads {THREADS} | smoke {SMOKE} | budget {TIME_BUDGET_S / 3600:.1f} h"
      f" | partitions {PARTITION_SEEDS}")

## 3 · Partition scheduler

Recipes register a target-free `prepare()` and a `fold_fn`. `prepare()` runs once and is cached. The
scheduler loops **partition → recipe → fold**. A recipe starts a new partition only if the time left
covers 1.25× its previous partition time, with a reserve kept for blending. A partition that cannot
finish is dropped as a whole.

In [ ]:
RESULTS: dict[str, dict] = {}
GROUPS: list[dict] = []


def register(title: str, prepare, fold_fn, first_partition_min: float = 30):
    """prepare() -> ctx ; fold_fn(ctx, k, tr, va) -> {name: (valid_pred, test_pred, info)}"""
    GROUPS.append(dict(title=title, prepare=prepare, fold_fn=fold_fn, ctx=None, last_s=None,
                       min_s=first_partition_min * 60, active=True))


def run_partition(group: dict, seed: int):
    need = 1.25 * group["last_s"] if group["last_s"] else group["min_s"]
    if remaining() < need + BLEND_RESERVE_S:
        print(f"[stop] {group['title']}: {remaining() / 60:.0f} min left, a partition needs ~{need / 60:.0f}")
        group["active"] = False
        return
    print(f"\n--- {group['title']} | partition seed {seed}   (start {elapsed()})")
    if group["ctx"] is None:
        t_prep = time.time()
        group["ctx"] = group["prepare"]()
        print(f"  features ready in {time.time() - t_prep:.0f}s")
    t_part = time.time()
    oof, tst, fold_auc = {}, {}, {}
    for k, (tr, va) in enumerate(FOLDS_BY_SEED[seed]):
        if remaining() < 0.5 * BLEND_RESERVE_S:
            print(f"  [abort] {group['title']}: out of time inside partition {seed}; partition discarded")
            group["active"] = False
            return
        t1 = time.time()
        for name, (pv, pt, info) in group["fold_fn"](group["ctx"], k, tr, va).items():
            oof.setdefault(name, np.zeros(len(y)))[va] = pv
            tst.setdefault(name, np.zeros(len(test_ids)))
            tst[name] += pt / NFOLD
            fold_auc.setdefault(name, []).append(roc_auc_score(y[va], pv))
            print(f"  fold {k} | {name:<22} auc {fold_auc[name][-1]:.6f} | {info} | {time.time() - t1:.0f}s")
        gc.collect()
    group["last_s"] = time.time() - t_part
    for name in oof:
        r = RESULTS.setdefault(name, dict(oofs=[], tests=[], seeds=[], partition_auc=[], fold_auc=None, seconds=0.0))
        r["oofs"].append(oof[name])
        r["tests"].append(tst[name])
        r["seeds"].append(seed)
        r["partition_auc"].append(float(roc_auc_score(y, oof[name])))
        r["fold_auc"] = r["fold_auc"] or fold_auc[name]
        r["seconds"] += group["last_s"]
        bag = roc_auc_score(y, np.mean(r["oofs"], axis=0))
        print(f"  >>> {name:<22} partition AUC {r['partition_auc'][-1]:.6f} | bag of {len(r['oofs'])}: {bag:.6f}")


def codes(values) -> np.ndarray:
    """Category identity as int32. TargetEncoder treats every input as categorical, so this matches
    string keys exactly and uses far less memory."""
    return pd.factorize(pd.Series(values), use_na_sentinel=False)[0].astype(np.int32)


def te_frame(K_tr, y_tr, K_others, smooths, names, style):
    """sklearn TargetEncoder (cross-fitted on the fit rows). style='per_col' interleaves the smoothing
    levels per column (najiama order); 'per_smooth' groups by smoothing (mikhailnaumov order)."""
    blocks_tr, blocks_ot = {}, [dict() for _ in K_others]
    for sm in smooths:
        tag = sm if isinstance(sm, str) else str(int(sm)) if float(sm).is_integer() else str(sm)
        enc = TargetEncoder(shuffle=True, cv=5, smooth=sm, random_state=42)
        a = enc.fit_transform(K_tr, y_tr).astype(np.float32)
        others = [enc.transform(K).astype(np.float32) for K in K_others]
        for i, col in enumerate(names):
            blocks_tr[(col, tag)] = a[:, i]
            for j, o in enumerate(others):
                blocks_ot[j][(col, tag)] = o[:, i]
    tags = [sm if isinstance(sm, str) else str(int(sm)) if float(sm).is_integer() else str(sm)
            for sm in smooths]
    order = ([(c, t) for c in names for t in tags] if style == "per_col"
             else [(c, t) for t in tags for c in names])
    fmt = "{}_TE_{}" if style == "per_col" else "{}_TE_s{}"
    cols = [fmt.format(c, t) for c, t in order]
    to_df = lambda blk, n: pd.DataFrame({cn: blk[key] for cn, key in zip(cols, order)}, index=range(n))
    return to_df(blocks_tr, len(K_tr)), [to_df(b, len(K)) for b, K in zip(blocks_ot, K_others)]

## 4 · chinzorigtganbat's leak-safe feature pipeline (input view for M)

Ported from the source. Target-free state is fitted once. Every target statistic (triple TE, income
neighbourhood channels over 8,192 and 16,384 bins) is cross-fitted with an inner KFold inside each
outer training fold.

In [ ]:
RAW = NUMS + CATS
CATEGORIES = {
    'Gender': ['Male', 'Female', 'Other'], 'City_Type': ['Urban', 'Suburban', 'Rural'],
    'Current_Car_Type': ['Sedan', 'SUV', 'Hatchback', 'Truck'],
    'Home_Charging_Possible': ['No', 'Yes'], 'Subsidy_Available': ['No', 'Yes'],
    'Range_Anxiety_Level': ['Low', 'Medium', 'High'],
}


@dataclass
class FeatureConfig:
    smooth_keys: bool = False
    digits: bool = True
    frequency: bool = True
    target_encoding: bool = True
    te_smoothing: tuple = ('auto', 10.0)
    te_columns: tuple = ()
    inner_folds: int = 5
    inner_seed: int = 17
    income_bins: tuple = ()
    neighborhood_smoothing: float = 10.0
    neighborhood_channels: tuple = ('central', 'neighbor', 'slope', 'curvature', 'count')
    income_kernel_sigma: float = 0.0
    scale_numeric: bool = False
    compact: bool = False


class FeaturePipeline:
    """Target-free state is fitted once; supervised encodings are cross-fitted on the fit rows."""

    def __init__(self, config: FeatureConfig):
        self.config = config

    def _base(self, X):
        c = self.config
        f = X[RAW].copy().reset_index(drop=True)
        for col in NUMS:
            f[col] = pd.to_numeric(f[col], errors='coerce').fillna(self.medians_[col])
        for col, levels in CATEGORIES.items():
            f[col] = f[col].map({v: i for i, v in enumerate(levels)}).fillna(-1).astype('int16')
        income = np.rint(f.Annual_Income_USD).astype('int64')
        commute = np.rint(10 * f.Daily_Commute_km).astype('int64')
        if c.digits:
            for power in range(5):
                f[f'income_digit_{power}'] = (income // 10 ** power) % 10
            f['commute_tenths'] = commute % 10
            f['commute_units'] = (commute // 10) % 10
        if c.compact:
            f['homecharge'] = f.Charging_Stations_Near_Home + 5 * f.Home_Charging_Possible
        if c.smooth_keys:
            f['smooth_key_income100'] = income // 100
            f['smooth_key_income1000'] = income // 1000
            f['smooth_key_commute_integer'] = commute // 10
        return f

    def _fit_supervised(self, base, yv):
        c = self.config
        state = {'prior': float(np.mean(yv)), 'encoders': [], 'bins': []}
        if c.target_encoding:
            keys = base[self.te_columns_].astype(str)
            for smoothing in c.te_smoothing:
                enc = TargetEncoder(smooth=smoothing, target_type='binary')
                enc.fit(keys, yv)
                state['encoders'].append(enc)
        income = base.Annual_Income_USD.to_numpy(float)
        for bins in c.income_bins:
            edges = self.income_edges_[bins]
            cds = np.searchsorted(edges[1:-1], income)
            sums = np.bincount(cds, weights=yv, minlength=bins)
            counts = np.bincount(cds, minlength=bins).astype(float)
            smooth = c.neighborhood_smoothing
            central = (sums + smooth * state['prior']) / (counts + smooth)
            weights = np.exp(-.5 * (np.arange(-2, 3) / .8) ** 2)
            neighbor = (convolve1d(sums, weights, mode='constant', cval=0) + smooth * state['prior']) / \
                       (convolve1d(counts, weights, mode='constant', cval=0) + smooth)
            left = np.r_[state['prior'], central[:-1]]
            right = np.r_[central[1:], state['prior']]
            values = dict(central=central, neighbor=neighbor, left=left, right=right, slope=right - left,
                          curvature=central - .5 * (left + right), count=np.log1p(counts))
            state['bins'].append((edges, values))
        if c.income_kernel_sigma > 0:
            vals = np.rint(income).astype('int64')
            lo, hi = int(vals.min()), int(vals.max())
            counts = np.bincount(vals - lo, minlength=hi - lo + 1).astype(float)
            sums = np.bincount(vals - lo, weights=yv, minlength=hi - lo + 1)
            radius = max(1, int(np.ceil(8 * c.income_kernel_sigma)))
            kernel = np.exp(-.5 * (np.arange(-radius, radius + 1) / c.income_kernel_sigma) ** 2)
            num = convolve1d(sums, kernel, mode='constant', cval=0)
            den = convolve1d(counts, kernel, mode='constant', cval=0)
            state['kernel'] = (lo, (num + c.neighborhood_smoothing * state['prior']) / (den + c.neighborhood_smoothing))
        return state

    def _supervised(self, base, state):
        blocks, names = [], []
        if state['encoders']:
            keys = base[self.te_columns_].astype(str)
            for i, enc in enumerate(state['encoders']):
                blocks.append(enc.transform(keys))
                names += [f'te{i}__{col}' for col in self.te_columns_]
        income = base.Annual_Income_USD.to_numpy(float)
        for q, (edges, values) in zip(self.config.income_bins, state['bins']):
            cds = np.searchsorted(edges[1:-1], income)
            for channel in self.config.neighborhood_channels:
                blocks.append(values[channel][cds, None])
                names.append(f'income_bin{q}__{channel}')
        if 'kernel' in state:
            lo, lookup = state['kernel']
            cds = np.rint(income).astype('int64') - lo
            valid = (cds >= 0) & (cds < len(lookup))
            result = np.full(len(base), state['prior'])
            result[valid] = lookup[cds[valid]]
            blocks.append(result[:, None])
            names.append('income_kernel')
        arr = np.column_stack(blocks).astype('float32') if blocks else np.empty((len(base), 0), dtype='float32')
        return pd.DataFrame(arr, columns=names)

    def _unsupervised(self, base):
        c = self.config
        columns = (['Annual_Income_USD', 'Daily_Commute_km', 'Charging_Stations_Near_Work',
                    'Environmental_Concern_Level', 'Home_Charging_Possible', 'Subsidy_Available',
                    'Range_Anxiety_Level', 'homecharge'] if c.compact else list(base.columns))
        out = base[columns].copy().astype('float32')
        if c.scale_numeric:
            out = (out - self.scale_mean_[columns]) / self.scale_std_[columns]
        if c.frequency:
            for col, mapping in self.frequencies_.items():
                out[f'freq__{col}'] = base[col].map(mapping).fillna(0).astype('float32')
        return out

    def fit_transform(self, X, yv):
        c = self.config
        self.medians_ = X[NUMS].median().fillna(0)
        base = self._base(X)
        income = base.Annual_Income_USD.to_numpy(float)
        lo, hi = float(income.min()), float(income.max())
        self.income_edges_ = {q: np.linspace(lo, max(lo + 1, hi), q + 1) for q in c.income_bins}
        self.te_columns_ = list(c.te_columns) if c.te_columns else [
            col for col in base if col in RAW or 'digit' in col or 'commute_' in col
            or 'income_mod_' in col or 'smooth_key_' in col]
        self.scale_mean_ = base.mean()
        self.scale_std_ = base.std(ddof=0).replace(0, 1).fillna(1)
        self.frequencies_ = {col: base[col].value_counts(normalize=True) for col in self.te_columns_} if c.frequency else {}
        self.supervised_state_ = self._fit_supervised(base, np.asarray(yv))
        template = self._supervised(base.iloc[:1], self.supervised_state_)
        encoded = np.empty((len(base), template.shape[1]), dtype='float32')
        for fit, hold in KFold(c.inner_folds, shuffle=True, random_state=c.inner_seed).split(base):
            state = self._fit_supervised(base.iloc[fit], np.asarray(yv)[fit])
            encoded[hold] = self._supervised(base.iloc[hold], state).to_numpy()
        result = pd.concat([self._unsupervised(base), pd.DataFrame(encoded, columns=template.columns)], axis=1)
        self.columns_ = [col for col in result if col in template.columns or result[col].nunique(dropna=False) > 1]
        return result[self.columns_]

    def transform(self, X):
        base = self._base(X)
        return pd.concat([self._unsupervised(base), self._supervised(base, self.supervised_state_)], axis=1)[self.columns_]


LGB_BASE = dict(objective='binary', metric='auc', learning_rate=.02, max_depth=5, num_leaves=31,
                min_data_in_leaf=10, feature_fraction=.30, lambda_l1=.071, lambda_l2=2.0, max_bin=1024,
                feature_pre_filter=False, verbosity=-1, seed=42, num_threads=THREADS)
CHINZO = {
    "C1_chinzo_lgbm": dict(
        features=FeatureConfig(income_bins=(8192, 16384), neighborhood_smoothing=1.0, smooth_keys=True,
                               te_smoothing=('auto', 10.0, 100.0)),
        learner="lightgbm", rounds=1377, params=dict(LGB_BASE, bagging_fraction=0.85, bagging_freq=1)),
    "C2_chinzo_catboost": dict(
        features=FeatureConfig(income_bins=(8192, 16384), smooth_keys=True, te_smoothing=('auto', 10.0, 100.0)),
        learner="catboost", rounds=944,
        params=dict(loss_function='Logloss', depth=5, learning_rate=0.04, l2_leaf_reg=5.0, rsm=0.5)),
    "C3_chinzo_linear": dict(
        features=FeatureConfig(digits=False, te_smoothing=(0.5,), te_columns=(
            'Age', 'Gender', 'Number_of_Cars_Owned', 'City_Type', 'Current_Car_Type', 'Range_Anxiety_Level'),
            neighborhood_smoothing=0.5, income_kernel_sigma=5.0, scale_numeric=True, compact=True),
        learner="lightgbm", rounds=967,
        params=dict(LGB_BASE, max_depth=-1, bagging_fraction=0.9, bagging_freq=1, lambda_l1=0.05, lambda_l2=10,
                    max_bin=1023, linear_tree=True, linear_lambda=0.5)),
}


def chinzo_member(name):
    cfg = CHINZO[name]

    def prepare():
        return dict(X=train_raw[RAW], Xt=test_raw[RAW])

    def fold(ctx, k, tr, va):
        prep = FeaturePipeline(cfg["features"])
        a = prep.fit_transform(ctx["X"].iloc[tr], y[tr])
        b, t = prep.transform(ctx["X"].iloc[va]), prep.transform(ctx["Xt"])
        rounds = max(20, int(cfg["rounds"] * R["fixed_scale"]))
        if cfg["learner"] == "lightgbm":
            m = lgb.train(cfg["params"], lgb.Dataset(a, label=y[tr]), num_boost_round=rounds)
            pv, pt = m.predict(b), m.predict(t)
        else:
            from catboost import CatBoostClassifier
            m = CatBoostClassifier(**cfg["params"], iterations=rounds, thread_count=THREADS, random_seed=42,
                                   verbose=False, allow_writing_files=False)
            m.fit(a, y[tr])
            pv, pt = m.predict_proba(b)[:, 1], m.predict_proba(t)[:, 1]
        return {name: (pv, pt, f"rounds {rounds} cols {a.shape[1]}")}

    return prepare, fold

## 5 · RealMLP core (yekenot port)

One network holds 8 parallel MLPs. Numerics get periodic PBLD embeddings; categoricals get one-hot or
embeddings. Training uses robust scaling with smooth clipping, per-group learning rates, label
smoothing and an EMA of the weights. The best-validation EMA epoch is kept, as in the source. The
yekenot feature view uses income/commute floors, quantile income bins, three categorical combos with
in-fold TE, and original-data income means.

In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.base import BaseEstimator, TransformerMixin

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device", TORCH_DEVICE)


def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)


class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X: np.ndarray, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X: np.ndarray, y=None) -> np.ndarray:
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X


class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens: int, cat_dims, embed_dim: int = 8, onehot_thresh: int = 8):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []
        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim, embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)

    def forward(self, x):
        batch_size, n_ens, _ = x.shape
        features = []
        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            encoded = torch.zeros(batch_size, n_ens, sum(onehot_dims), device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                encoded.scatter_(2, onehot_x[:, :, idx:idx + 1].long() + start, 1.0)
                start += dim
            features.append(encoded)
        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = [emb_list[m](x[:, m, feat_idx:feat_idx + 1].long()) for m in range(self.n_ens)]
            features.append(torch.cat(feat_embs, dim=1))
        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens: int, n_features: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    def __init__(self, n_ens: int, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        return x + self.bias if self.bias is not None else x


class ResidualBlock(nn.Module):
    def __init__(self, n_ens: int, dim: int, dropout: float, activation=nn.SiLU):
        super().__init__()
        self.linear = NTPLinear(n_ens=n_ens, in_features=dim, out_features=dim)
        self.act = activation()
        self.drop = nn.Dropout(dropout)
        self.res_scale = nn.Parameter(torch.ones(n_ens, dim) * 0.1)

    def forward(self, x):
        return x + self.drop(self.act(self.linear(x))) * self.res_scale.unsqueeze(0)


class PBLDEmbedding(nn.Module):
    """Periodic Basis with Learned Decay embedding for numerical features."""

    def __init__(self, n_ens: int, n_features: int, hidden_dim: int = 16, out_dim: int = 4,
                 freq_scale: float = 0.1, activation=nn.GELU):
        super().__init__()
        self.w1 = nn.Parameter(torch.empty(n_ens, n_features, hidden_dim))
        nn.init.normal_(self.w1, mean=0.0, std=freq_scale / math.sqrt(hidden_dim))
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim))
        self.b2 = nn.Parameter(torch.zeros(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        periodic = torch.cos(2 * math.pi * (x.unsqueeze(-1) * self.w1.unsqueeze(0) + self.b1.unsqueeze(0)))
        transformed = self.act(torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2) + self.b2.unsqueeze(0))
        return torch.cat([x.unsqueeze(-1), transformed], dim=-1).flatten(start_dim=2)


class RealMLP(nn.Module):
    def __init__(self, output_dim: int, cat_dims, n_numerical: int, cfg: dict):
        super().__init__()
        n_ens, embed_dim = cfg["n_ens"], cfg["embed_dim"]
        self.n_ens = n_ens
        self.cate = CategoricalFeatureLayer(n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim,
                                            onehot_thresh=cfg["onehot_thresh"])
        self.num_embed = PBLDEmbedding(n_ens=n_ens, n_features=n_numerical, hidden_dim=cfg["pbld_hidden_dim"],
                                       out_dim=cfg["pbld_out_dim"], freq_scale=cfg["pbld_freq_scale"],
                                       activation=cfg["pbld_activation"])
        total_dim = n_numerical * cfg["pbld_out_dim"] + sum(
            c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        hidden_dims, act = cfg["hidden_dims"], cfg["activation"]
        self._dropout_modules = []
        layers = [ScalingLayer(n_ens=n_ens, n_features=total_dim)] if cfg["add_front_scale"] else []
        self.first_linear = NTPLinear(n_ens=n_ens, in_features=total_dim, out_features=hidden_dims[0])
        layers.extend([self.first_linear, act()])
        in_dim = hidden_dims[0]
        for hdim in hidden_dims[1:]:
            if in_dim != hdim:
                layers.extend([NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=hdim), act()])
                in_dim = hdim
            block = ResidualBlock(n_ens=n_ens, dim=hdim, dropout=cfg["dropout"], activation=act)
            self._dropout_modules.append(block.drop)
            layers.append(block)
        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)
        with torch.no_grad():
            self.output_layer.weight.mul_(0.1)
            self.output_layer.bias.zero_()

    def forward(self, x_num, x_cat):
        x_num = self.num_embed(x_num.unsqueeze(1).expand(-1, self.n_ens, -1))
        x_cat = self.cate(x_cat.unsqueeze(1).expand(-1, self.n_ens, -1))
        return self.output_layer(self.hidden(torch.cat([x_num, x_cat], dim=2)))


def apply_schedule(init_value: float, progress: float, sched: str, flat_ratio: float = 0.3) -> float:
    if sched == "constant":
        return init_value
    if sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    if sched in ("flat_cos", "flat_anneal"):
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * ((math.cos(math.pi * t) + 1) / 2 if sched == "flat_cos" else (1 - t))
    if sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    if sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    raise ValueError(f"Unknown schedule: '{sched}'")


def get_parameter_groups(model: RealMLP, p: dict):
    first_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    LR, WD = p["lr"], p["weight_decay"]
    return [
        {"params": scale_p, "lr": LR * p["lr_scale_mult"], "weight_decay": WD * p["wd_scale_mult"]},
        {"params": pbld_p, "lr": LR * p["pbld_lr_factor"], "weight_decay": WD},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay": WD * p["first_layer_wd_factor"]},
        {"params": other_w_p, "lr": LR, "weight_decay": WD},
        {"params": bias_p, "lr": LR * p["lr_bias_mult"], "weight_decay": WD * p["wd_bias_mult"]},
    ]


def binary_bce_loss(y_true, logits, ls: float = 0.0):
    if ls > 0.0:
        y_true = y_true * (1.0 - ls) + 0.5 * ls
    return ((1.0 - y_true) * logits + F.softplus(-logits)).mean()


class RealMLPClassifier:
    """Fit on (train, valid); keeps the best-validation-epoch EMA weights, as in the source."""

    def __init__(self, cfg: dict):
        self.p = cfg

    def fit(self, X_train, y_train, X_val, y_val, cat_col_names):
        p, dev = self.p, TORCH_DEVICE
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]
        X_tr_num = X_train[num_col_names].to_numpy(np.float32)
        X_val_num = X_val[num_col_names].to_numpy(np.float32)
        X_tr_cat = X_train[cat_col_names].to_numpy(np.int64)
        X_val_cat = X_val[cat_col_names].to_numpy(np.int64)
        self.preprocessor_ = NumericalPreprocessor(p["tfms"]).fit(X_tr_num)
        X_tr_num, X_val_num = self.preprocessor_.transform(X_tr_num), self.preprocessor_.transform(X_val_num)
        self.cat_col_names_, self.num_col_names_ = cat_col_names, num_col_names
        self.cat_dims_ = (np.concatenate([X_tr_cat, X_val_cat]).max(axis=0) + 1).tolist()
        cat_max = np.array(self.cat_dims_) - 1
        X_tr_cat, X_val_cat = np.clip(X_tr_cat, 0, cat_max), np.clip(X_val_cat, 0, cat_max)
        self.model_ = RealMLP(1, self.cat_dims_, X_tr_num.shape[1], p).to(dev)
        groups = get_parameter_groups(self.model_, p)
        for g in groups:
            g["lr_base"] = g["lr"]
        opt = torch.optim.AdamW(groups, betas=(p["mom"], p["sq_mom"]))
        Xtn = torch.as_tensor(X_tr_num, device=dev)
        Xtc = torch.as_tensor(X_tr_cat, device=dev)
        ytt = torch.as_tensor(np.asarray(y_train), dtype=torch.float32, device=dev)
        Xvn = torch.as_tensor(X_val_num, device=dev)
        Xvc = torch.as_tensor(X_val_cat, device=dev)
        n, n_ens, bs = len(ytt), p["n_ens"], p["train_bs"]
        total_steps, order = p["epochs"] * n, np.arange(n)
        best_score, self.best_epoch_, best_state = -np.inf, 0, None
        ema = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
        for epoch in range(p["epochs"]):
            self.model_.train()
            for start in range(0, n, bs):
                progress = (epoch * n + start) / total_steps
                idx = order[start:start + bs]
                for g in opt.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, p["lr_sched"], p["flat_ratio"])
                opt.zero_grad()
                out = self.model_(Xtn[idx], Xtc[idx])
                ls_val = apply_schedule(p["ls_eps"], progress, p["ls_eps_sched"], p["flat_ratio"])
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"], p["flat_ratio"])
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val
                loss = binary_bce_loss(ytt[idx].repeat_interleave(n_ens), out.reshape(-1), ls=ls_val)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                opt.step()
                with torch.no_grad():
                    for key, value in self.model_.state_dict().items():
                        if torch.is_floating_point(value):
                            ema[key].mul_(p["ema_decay"]).add_(value.detach(), alpha=1.0 - p["ema_decay"])
                        else:
                            ema[key].copy_(value)
            np.random.shuffle(order)
            live = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
            self.model_.load_state_dict(ema, strict=True)
            val = self._predict_tensors(Xvn, Xvc)
            score = roc_auc_score(np.asarray(y_val), val)
            if score > best_score:
                best_score, self.best_epoch_, self.best_val_ = score, epoch + 1, val
                best_state = {k: v.detach().clone() for k, v in ema.items()}
            self.model_.load_state_dict(live, strict=True)
        self.model_.load_state_dict(best_state, strict=True)
        self.best_score_ = best_score
        return self

    def _predict_tensors(self, Xn, Xc):
        self.model_.eval()
        bs = self.p["eval_bs"]
        with torch.no_grad():
            return np.concatenate([torch.sigmoid(self.model_(Xn[s:s + bs], Xc[s:s + bs])).mean(dim=1)
                                   .squeeze(-1).cpu().numpy() for s in range(0, len(Xn), bs)])

    def predict_proba1(self, X):
        Xn = self.preprocessor_.transform(X[self.num_col_names_].to_numpy(np.float32))
        Xc = np.clip(X[self.cat_col_names_].to_numpy(np.int64), 0, np.array(self.cat_dims_) - 1)
        return self._predict_tensors(torch.as_tensor(Xn, device=TORCH_DEVICE),
                                     torch.as_tensor(Xc, device=TORCH_DEVICE))


REALMLP_CONFIG = {
    "n_ens": 8, "embed_dim": 6, "onehot_thresh": 4, "hidden_dims": [256, 256, 256],
    "dropout": 0.05, "p_drop_sched": "expm4t", "activation": nn.SiLU, "add_front_scale": True,
    "pbld_hidden_dim": 20, "pbld_out_dim": 5, "pbld_freq_scale": 5.0, "pbld_activation": nn.PReLU,
    "pbld_lr_factor": 0.093,
    "lr": 0.01, "mom": 0.9, "sq_mom": 0.99, "lr_sched": "flat_anneal", "flat_ratio": 0.3,
    "first_layer_lr_factor": 1.2, "first_layer_wd_factor": 0.1, "lr_scale_mult": 10.0, "lr_bias_mult": 0.1,
    "weight_decay": 0.013, "wd_scale_mult": 0.1, "wd_bias_mult": 0.5, "ema_decay": 0.997875, "grad_clip": 1.0,
    "ls_eps": 0.04, "ls_eps_sched": "cos",
    "tfms": ["median_center", "robust_scale", "smooth_clip"],
    "epochs": 2, "train_bs": 256, "eval_bs": 10240,
}
if SMOKE:
    REALMLP_CONFIG.update(epochs=1, n_ens=2, hidden_dims=[32, 32], train_bs=512)

REALMLP_COMBOS = [("Annual_Income_USD", "Range_Anxiety_Level"), ("Age", "Range_Anxiety_Level"),
                  ("Annual_Income_USD", "Income_/_100_floor_")]


def prepare_realmlp():
    """The source's feature_engineering(): factorize maps are fitted on train, applied to test."""
    X, Xt = train_raw.drop(columns=[ID, TARGET]).copy(), test_raw.drop(columns=[ID]).copy()
    og = orig_raw.copy() if orig_raw is not None else None
    if og is not None:
        og[TARGET] = og[TARGET].map({"No": 0, "Yes": 1})
    cat_cols, num_cols = list(CATS), list(NUMS)
    cmap = {}

    def fe(df, fit):
        for col in cat_cols:
            df[col] = df[col].fillna("missing")
        for col in num_cols:
            df[col] = df[col].fillna(0.0)
        for col in cat_cols:
            if fit:
                c, cmap[col] = df[col].factorize()
            else:
                c = df[col].map({v: i for i, v in enumerate(cmap[col])}).fillna(-1).astype("int32")
            df[col] = pd.Series(c, index=df.index).astype("category")
        df["_Daily_Commute_km_/_Age"] = (df["Daily_Commute_km"] / (df["Age"] + 1e-6)).astype("float32")
        df["Income_/_100_floor_"] = np.floor(df["Annual_Income_USD"] / 100.0).astype("float32").astype("category")
        df["Income_/_1000_floor_"] = np.floor(df["Annual_Income_USD"] / 1000.0).astype("float32").astype("category")
        df["Income_/_10000_floor_"] = np.floor(df["Annual_Income_USD"] / 10000.0).astype("float32").astype("category")
        df["Daily_km_/_5_floor_"] = np.floor(df["Daily_Commute_km"] / 5.0).astype("float32").astype("category")
        for col in [c for c in num_cols if c not in ("Annual_Income_USD", "Charging_Stations_Near_Home")]:
            if fit:
                c, cmap[col] = np.floor(df[col]).factorize()
            else:
                c = np.floor(df[col]).map({v: i for i, v in enumerate(cmap[col])}).fillna(-1).astype("int32")
            df[f"{col}_cat_"] = pd.Series(c, index=df.index).astype("category")
        df["_Daily_Commute_km_decimal"] = (df["Daily_Commute_km"] % 1).round(2).astype("float32")
        df["Annual_Income_USD_is_multiple_10_"] = (np.floor(df["Annual_Income_USD"]) % 10 == 0).astype("category")
        if og is not None:
            df["_Annual_Income_USD_mean_target_orig"] = (
                df["Annual_Income_USD"].map(og.groupby("Annual_Income_USD")[TARGET].mean())
                .fillna(og[TARGET].mean()).astype("float32"))
        if fit:
            cmap["_count"] = df["Annual_Income_USD"].value_counts()
        df["_Annual_Income_USD_count"] = df["Annual_Income_USD"].astype(object).map(cmap["_count"]).fillna(0).astype("int32")
        for n_bins in (400, 600, 800, 900, 1100):
            name = f"Annual_Income_USD_{n_bins}_quantile_bin_"
            if fit:
                cmap[name] = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile",
                                              subsample=None).fit(df[["Annual_Income_USD"]])
            df[name] = pd.Series(cmap[name].transform(df[["Annual_Income_USD"]]).ravel().astype("int32"),
                                 index=df.index).astype("category")
        for cols in REALMLP_COMBOS:
            name = "_".join(cols) + "_"
            combo = df[cols[0]].astype(str)
            for col in cols[1:]:
                combo = combo + "_" + df[col].astype(str)
            if fit:
                c, cmap[name] = pd.factorize(combo, sort=False)
            else:
                c = combo.map({v: i for i, v in enumerate(cmap[name])}).fillna(-1).astype("int32")
            df[name] = pd.Series(c, index=df.index).astype("category")
        return df

    X, Xt = fe(X, True), fe(Xt, False)
    cats = cat_cols + [c for c in X.columns if c.endswith("_")]
    # Categorical *values* are used as indices (for float floors: the floor value itself), as in the source.
    for frame in (X, Xt):
        for c in cats:
            frame[c] = frame[c].astype("float64").fillna(-1).astype("int64")
    print(f"  RealMLP view: {X.shape[1]} columns, {len(cats)} categorical")
    return dict(X=X, Xt=Xt, cats=cats, combos=["_".join(c) + "_" for c in REALMLP_COMBOS])


def fold_realmlp(ctx, k, tr, va):
    A, B, C = ctx["X"].iloc[tr].reset_index(drop=True), ctx["X"].iloc[va].reset_index(drop=True), ctx["Xt"].copy()
    enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)
    ea = enc.fit_transform(A[ctx["combos"]].to_numpy(), y[tr])
    eb, ec = enc.transform(B[ctx["combos"]].to_numpy()), enc.transform(C[ctx["combos"]].to_numpy())
    for i, c in enumerate(ctx["combos"]):
        A[f"_{c}TE"], B[f"_{c}TE"], C[f"_{c}TE"] = ea[:, i], eb[:, i], ec[:, i]
    seed_everything(42 + 101 * k + int(va[:7].sum() % 9973))   # differs per partition and fold
    model = RealMLPClassifier(REALMLP_CONFIG).fit(A, y[tr], B, y[va], cat_col_names=ctx["cats"])
    pv, pt = model.best_val_, model.predict_proba1(C)
    info = f"best epoch {model.best_epoch_} cols {A.shape[1]}"
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"N_realmlp": (pv, pt, info)}




def realmlp_fold(name: str, cfg: dict, seed_offset: int):
    """Recipe on the yekenot feature view; the seed differs per partition, fold and variant."""

    def fold(ctx, k, tr, va):
        A, B = ctx["X"].iloc[tr].reset_index(drop=True), ctx["X"].iloc[va].reset_index(drop=True)
        C = ctx["Xt"].copy()
        enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)
        ea = enc.fit_transform(A[ctx["combos"]].to_numpy(), y[tr])
        eb, ec = enc.transform(B[ctx["combos"]].to_numpy()), enc.transform(C[ctx["combos"]].to_numpy())
        for i, c in enumerate(ctx["combos"]):
            A[f"_{c}TE"], B[f"_{c}TE"], C[f"_{c}TE"] = ea[:, i], eb[:, i], ec[:, i]
        seed_everything(seed_offset + 101 * k + int(va[:7].sum() % 9973))
        model = RealMLPClassifier(cfg).fit(A, y[tr], B, y[va], cat_col_names=ctx["cats"])
        pv, pt = model.best_val_, model.predict_proba1(C)
        info = f"best epoch {model.best_epoch_}/{cfg['epochs']} cols {A.shape[1]}"
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {name: (pv, pt, info)}

    return fold


N2_CONFIG = dict(REALMLP_CONFIG, epochs=4)
N3_CONFIG = dict(REALMLP_CONFIG, hidden_dims=[512, 512, 512], dropout=0.1, epochs=3)
M_CONFIG = dict(REALMLP_CONFIG)
if SMOKE:
    N2_CONFIG.update(epochs=1)
    N3_CONFIG.update(epochs=1, hidden_dims=[32, 32])

## 6 · New neural recipes and the loaded components

**M** runs the RealMLP on the chinzorigtganbat pipeline output. The 6 raw categoricals stay
categorical (one-hot); everything else, including the cross-fitted TE and neighbourhood channels, is
numeric and gets PBLD embeddings.

In [ ]:
M_FEATURES = FeatureConfig(income_bins=(8192, 16384), neighborhood_smoothing=1.0, smooth_keys=True,
                           te_smoothing=("auto", 10.0, 100.0))


def prepare_teview():
    return dict(X=train_raw[RAW], Xt=test_raw[RAW])


def fold_teview(ctx, k, tr, va):
    prep = FeaturePipeline(M_FEATURES)
    A = prep.fit_transform(ctx["X"].iloc[tr], y[tr]).reset_index(drop=True)
    B, C = prep.transform(ctx["X"].iloc[va]), prep.transform(ctx["Xt"])
    cats = [c for c in CATEGORIES if c in A.columns]
    seed_everything(7_000 + 101 * k + int(va[:7].sum() % 9973))
    model = RealMLPClassifier(M_CONFIG).fit(A, y[tr], B, y[va], cat_col_names=cats)
    pv, pt = model.best_val_, model.predict_proba1(C)
    info = f"best epoch {model.best_epoch_}/{M_CONFIG['epochs']} cols {A.shape[1]}"
    del model, prep
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"M_realmlp_teview": (pv, pt, info)}


register("Recipe N2 · RealMLP 4 epochs", prepare_realmlp, realmlp_fold("N2_realmlp_e4", N2_CONFIG, 3_000),
         first_partition_min=30)
register("Recipe M · RealMLP on the nested-TE view", prepare_teview, fold_teview, first_partition_min=30)
register("Recipe N3 · wide RealMLP", prepare_realmlp, realmlp_fold("N3_realmlp_wide", N3_CONFIG, 5_000),
         first_partition_min=35)

# Bagged outputs of the author's earlier public notebooks (same data, same fold partitions).
PRIOR_RUNS = {
    "ev-quantum-forge-fusion-iii-realmlp-s6e9":
        ["C1_chinzo_lgbm", "A1_najiama_xgb", "D_megayak_liftwin", "C2_chinzo_catboost", "N_realmlp"],
    "ev-quantum-forge-fusion-ii-partition-bagged-s6e9":
        ["A2_najiama_lgbm", "C3_chinzo_linear", "E_bulsah_mrte"],
}


def find_prior(slug: str, fname: str) -> str | None:
    local = dict(x.split("=", 1) for x in os.environ.get("FUSION_PRIOR", "").split(";") if "=" in x)
    if slug in local:
        path = Path(local[slug]) / fname
        return str(path) if path.exists() else None
    hits = [h for h in glob.glob(f"/kaggle/input/**/{fname}", recursive=True) if slug in h.lower()]
    return hits[0] if hits else None


for slug, comp_names in PRIOR_RUNS.items():
    oof_path, test_path = find_prior(slug, "oof_components.csv"), find_prior(slug, "test_components.csv")
    if not (oof_path and test_path):
        print(f"[warn] outputs of {slug} not attached; its components are skipped")
        continue
    o = train_raw[[ID]].merge(pd.read_csv(oof_path, usecols=[ID, "fold"] + comp_names), on=ID, how="left")
    t = pd.DataFrame({ID: test_ids}).merge(pd.read_csv(test_path, usecols=[ID] + comp_names), on=ID, how="left")
    assert not o.isna().any().any() and not t.isna().any().any(), f"{slug}: id mismatch"
    if not SMOKE:
        assert (o["fold"].to_numpy() == fold_id).all(), f"{slug}: fold partition differs"
    for n in comp_names:
        ov = o[n].to_numpy(float)
        RESULTS[n] = dict(oofs=[ov], tests=[t[n].to_numpy(float)], seeds=list(PARTITION_SEEDS),
                          partition_auc=[float(roc_auc_score(y, ov))], seconds=0.0, loaded=slug,
                          fold_auc=[float(roc_auc_score(y[va], ov[va])) for _, va in GATE_FOLDS])
        print(f"loaded {n:<20} OOF AUC {RESULTS[n]['partition_auc'][0]:.6f}  <- {slug}")

## 7 · Run: partition → recipe → fold

In [ ]:
for p_idx, seed in enumerate(PARTITION_SEEDS):
    print(f"\n{'#' * 90}\nPARTITION {p_idx + 1}/{len(PARTITION_SEEDS)} · fold seed {seed}   ({elapsed()})\n{'#' * 90}")
    for group in GROUPS:
        if group["active"]:
            run_partition(group, seed)
for group in GROUPS:
    group["ctx"] = None
gc.collect()
for name, r in RESULTS.items():
    r["oof"] = np.mean(r["oofs"], axis=0)
    r["test"] = np.mean(r["tests"], axis=0)
    r["auc"] = float(roc_auc_score(y, r["oof"]))

## 8 · Component scoreboard

In [ ]:
if not RESULTS:
    raise RuntimeError("No component finished inside the budget")
names = sorted(RESULTS, key=lambda n: -RESULTS[n]["auc"])
board = pd.DataFrame([{"component": n, "source": RESULTS[n].get("loaded", "trained here"),
                       "oof_auc": RESULTS[n]["auc"],
                       "fold_min": min(RESULTS[n]["fold_auc"]), "fold_max": max(RESULTS[n]["fold_auc"]),
                       "partitions": len(RESULTS[n]["oofs"]),
                       "single_partition_mean": float(np.mean(RESULTS[n]["partition_auc"])),
                       "minutes": RESULTS[n]["seconds"] / 60} for n in names])
print(board.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
P = np.column_stack([RESULTS[n]["oof"] for n in names])
PT = np.column_stack([RESULTS[n]["test"] for n in names])
sub_rows = np.random.default_rng(0).choice(len(y), min(200_000, len(y)), replace=False)
rank_corr = pd.DataFrame(np.corrcoef(np.apply_along_axis(rankdata, 0, P[sub_rows]), rowvar=False),
                         index=names, columns=names)
print("\nOOF Spearman correlation (200k-row sample):")
print(rank_corr.round(4).to_string())

## 9 · Nested-gated logit blend

Caruana ensemble selection with replacement, on logits. For each outer fold the weights are chosen on
the other four folds and scored on the held-out fold, against the best single component chosen the
same way. The pooled weights ship only if the blend wins at least 4 of 5 folds with a positive mean
gain. Otherwise the best single component ships.

In [ ]:
Lg = logit(np.clip(P, 1e-7, 1 - 1e-7))
LgT = logit(np.clip(PT, 1e-7, 1 - 1e-7))


def fast_auc(yv, s):
    r = rankdata(s)
    npos = yv.sum()
    return (r[yv == 1].sum() - npos * (npos + 1) / 2) / (npos * (len(yv) - npos))


def hill_climb(L, yv, steps=60):
    k = L.shape[1]
    single = np.array([fast_auc(yv, L[:, j]) for j in range(k)])
    counts = np.zeros(k)
    counts[single.argmax()] = 1
    cur, cur_auc = L[:, single.argmax()].copy(), single.max()
    for _ in range(steps):
        n = counts.sum()
        trial = [fast_auc(yv, (cur * n + L[:, j]) / (n + 1)) for j in range(k)]
        j = int(np.argmax(trial))
        if trial[j] <= cur_auc + 1e-7:
            break
        counts[j] += 1
        cur, cur_auc = (cur * n + L[:, j]) / (n + 1), trial[j]
    return counts / counts.sum(), single


gate = []
for k, (tr, va) in enumerate(GATE_FOLDS):
    w, single = hill_climb(Lg[tr], y[tr])
    j = int(single.argmax())
    b_auc = fast_auc(y[va], Lg[va] @ w)
    s_auc = fast_auc(y[va], Lg[va, j])
    gate.append(dict(fold=k, blend_auc=b_auc, best_single=names[j], single_auc=s_auc, gain=b_auc - s_auc,
                     weights={names[i]: round(float(v), 3) for i, v in enumerate(w) if v > 0}))
gate_df = pd.DataFrame(gate)
print(gate_df.drop(columns="weights").to_string(index=False, float_format=lambda v: f"{v:.6f}"))
wins = int((gate_df.gain > 0).sum())
use_blend = wins >= 4 and gate_df.gain.mean() > 0
W, single_all = hill_climb(Lg, y)
if use_blend:
    final_oof, final_test = Lg @ W, LgT @ W
    chosen = {names[i]: float(v) for i, v in enumerate(W) if v > 0}
else:
    j = int(single_all.argmax())
    final_oof, final_test = Lg[:, j], LgT[:, j]
    chosen = {names[j]: 1.0}
print(f"\nnested gate: blend wins {wins}/5, mean gain {gate_df.gain.mean():+.6f} -> "
      f"{'BLEND' if use_blend else 'BEST SINGLE'}")
print("final weights:", {k_: round(v, 3) for k_, v in chosen.items()})
print(f"final OOF AUC {roc_auc_score(y, final_oof):.6f} | equal-weight logit mean "
      f"{roc_auc_score(y, Lg.mean(1)):.6f} | best single {single_all.max():.6f}")

## 10 · Export

In [ ]:
pred = expit(final_test)
assert len(pred) == len(test_ids) and np.isfinite(pred).all()
submission = sample[[ID]].merge(pd.DataFrame({ID: test_ids, TARGET: pred}), on=ID, how="left")
assert submission[TARGET].notna().all() and len(submission) == len(sample)
submission.to_csv("submission.csv", index=False)

oof_table = pd.DataFrame({ID: train_raw[ID], "fold": fold_id, TARGET: y,
                          **{n: RESULTS[n]["oof"].astype("float32") for n in names}, "fusion": expit(final_oof)})
test_table = pd.DataFrame({ID: test_ids, **{n: RESULTS[n]["test"].astype("float32") for n in names}, "fusion": pred})
oof_table.to_csv(OUT / "oof_components.csv", index=False, float_format="%.7f")
test_table.to_csv(OUT / "test_components.csv", index=False, float_format="%.7f")
report = dict(components=board.to_dict("records"), gate=gate, blend_used=bool(use_blend), weights=chosen,
              final_oof_auc=float(roc_auc_score(y, final_oof)), minutes=(time.time() - T0) / 60,
              gpu=GPU, smoke=SMOKE, folds=NFOLD, partition_seeds=list(PARTITION_SEEDS))
(OUT / "fusion_report.json").write_text(json.dumps(report, indent=2, default=float))
print(submission.head())
print(f"\nwrote submission.csv ({len(submission):,} rows) | total {elapsed()}")

## Takeaways

- The bagged tree recipes and the first RealMLP come from Fusion II and III. The three new neural
  views were trained here on the same partitions.
- The correlation table shows which view differs most from the trees; the gate decides whether it ships.
- `fusion_outputs/` holds every component's OOF and test vector, so the next challenger can build on it.